# ROGII - Wellbore Geology Prediction - Stage 5 submission notebook

Self-contained: no internet access needed (Code Competition requirement).

**Stage 5: ensemble.** Non-negative least-squares blend of the three real, working
signals - `linear_prior` (Stage 2), `windowed_match` (Stage 3), and Stage 4a's
`HistGradientBoostingRegressor`. The CNN from Stage 4b is excluded (public LB 72.734,
clearly worse than Stage 4a alone - see `../context/05-plan-of-attack.md`).

Blend weights were fit on leak-free Stage 4a out-of-fold predictions across all 773
training wells, then verified with a held-out GroupKFold check (weights fit on 4/5 of
wells, evaluated on the other 1/5 - not just in-sample):

- In-sample blend RMSE: 51.72
- **Held-out blend RMSE: 52.03** (the honest number) vs Stage 4a alone's 52.90 local /
  45.196 public LB - a real, modest improvement (~1.6%).
- Fitted weights: `linear_prior` 0.123, `windowed_match` 0.096, `stage4a_pred` 0.781 -
  Stage 4a dominates, but the other two signals still contribute positively rather than
  being zeroed out.

In [ ]:
import glob
import os
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

RANDOM_STATE = 42
HALF_WINDOW = 5
SEARCH_EXTRA_FT = 100.0
ACCEPT_SLACK = 1.5

# Blend weights from src/stage5_ensemble.py's NNLS fit on the full 773-well OOF set,
# verified to generalize via a separate held-out GroupKFold check (52.03 RMSE).
BLEND_WEIGHTS = {"linear_prior": 0.1232, "windowed_match": 0.0960, "stage4a_pred": 0.7809}

_KAGGLE_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
DATA_DIR = _KAGGLE_DIR if os.path.isdir(_KAGGLE_DIR) else os.path.join("..", "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"DATA_DIR = {DATA_DIR}")
assert os.path.isdir(TRAIN_DIR), f"train dir not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"test dir not found: {TEST_DIR}"

In [ ]:
def list_wells(split_dir):
    files = glob.glob(os.path.join(split_dir, "*__horizontal_well.csv"))
    return sorted(os.path.basename(f).split("__")[0] for f in files)


def linear_prior_predict(known, eval_rows):
    if len(known) < 2:
        fallback = known["TVT_input"].mean() if len(known) else np.nan
        return np.full(len(eval_rows), fallback)
    model = LinearRegression()
    model.fit(known[["MD", "Z"]], known["TVT_input"])
    return model.predict(eval_rows[["MD", "Z"]])


def windowed_shape_match(prior_preds, eval_gr, tw_tvt, tw_gr,
                          half_win=HALF_WINDOW, search_extra=SEARCH_EXTRA_FT,
                          accept_slack=ACCEPT_SLACK):
    n = len(prior_preds)
    if n == 0:
        return prior_preds
    gr_filled = pd.Series(eval_gr).interpolate(limit_direction="both").to_numpy()
    if np.isnan(gr_filled).all():
        return prior_preds
    refined = prior_preds.copy()
    match_err = np.full(n, np.nan)
    for i in range(n):
        lo_row, hi_row = max(0, i - half_win), min(n, i + half_win + 1)
        local_gr = gr_filled[lo_row:hi_row]
        L = len(local_gr)
        if np.isnan(local_gr).any() or L < 3:
            continue
        center_prior = prior_preds[i]
        lo_idx = np.searchsorted(tw_tvt, center_prior - search_extra)
        hi_idx = np.searchsorted(tw_tvt, center_prior + search_extra)
        if hi_idx - lo_idx < L:
            continue
        seg_gr = tw_gr[lo_idx:hi_idx]
        seg_tvt = tw_tvt[lo_idx:hi_idx]
        windows = np.lib.stride_tricks.sliding_window_view(seg_gr, L)
        sse = np.sum((windows - local_gr[None, :]) ** 2, axis=1)
        best = int(np.argmin(sse))
        center_offset = i - lo_row
        refined[i] = seg_tvt[best + center_offset]
        match_err[i] = sse[best] / L
    valid = ~np.isnan(match_err)
    if valid.sum() == 0:
        return prior_preds
    thresh = np.nanmedian(match_err[valid]) * accept_slack
    keep = valid & (match_err <= thresh)
    return np.where(keep, refined, prior_preds)


FEATURE_COLS = [
    "MD", "X", "Y", "Z", "GR", "linear_prior", "windowed_match",
    "match_minus_prior", "dist_from_known_boundary", "eval_zone_frac",
    "known_zone_rows",
]


def build_well_features(well, split_dir, has_target):
    hz = pd.read_csv(os.path.join(split_dir, f"{well}__horizontal_well.csv")).reset_index(drop=True)
    tw_path = os.path.join(split_dir, f"{well}__typewell.csv")
    tw = pd.read_csv(tw_path).dropna(subset=["TVT", "GR"]).sort_values("TVT")

    known = hz[hz["TVT_input"].notna()]
    eval_rows = hz[hz["TVT_input"].isna()]
    if len(eval_rows) == 0:
        return None

    linear_prior = linear_prior_predict(known, eval_rows)

    if len(tw) >= HALF_WINDOW * 2 + 1:
        tw_tvt = tw["TVT"].to_numpy()
        tw_gr = tw["GR"].to_numpy()
        eval_gr = eval_rows["GR"].to_numpy()
        windowed_match = windowed_shape_match(linear_prior, eval_gr, tw_tvt, tw_gr)
    else:
        windowed_match = linear_prior.copy()

    known_md_max = known["MD"].max() if len(known) else eval_rows["MD"].min()
    n_eval = len(eval_rows)

    df = pd.DataFrame({
        "well": well,
        "row_idx": eval_rows.index.to_numpy(),
        "MD": eval_rows["MD"].to_numpy(),
        "X": eval_rows["X"].to_numpy(),
        "Y": eval_rows["Y"].to_numpy(),
        "Z": eval_rows["Z"].to_numpy(),
        "GR": eval_rows["GR"].to_numpy(),
        "linear_prior": linear_prior,
        "windowed_match": windowed_match,
        "match_minus_prior": windowed_match - linear_prior,
        "dist_from_known_boundary": eval_rows["MD"].to_numpy() - known_md_max,
        "eval_zone_frac": (np.arange(n_eval) + 1) / n_eval,
        "known_zone_rows": len(known),
    })

    if has_target and "TVT" in hz.columns:
        df["target"] = eval_rows["TVT"].to_numpy()

    return df


def build_dataset(split_dir, wells, has_target):
    frames, failed = [], []
    for well in wells:
        try:
            f = build_well_features(well, split_dir, has_target)
            if f is not None:
                frames.append(f)
        except Exception as exc:  # noqa: BLE001
            print(f"  well {well} failed ({exc}); skipping")
            failed.append(well)
    if not frames:
        return pd.DataFrame(), failed
    return pd.concat(frames, ignore_index=True), failed

In [ ]:
t0 = time.time()
train_wells = list_wells(TRAIN_DIR)
print(f"Building TRAIN features for {len(train_wells)} wells...")
train_data, train_failed = build_dataset(TRAIN_DIR, train_wells, has_target=True)
train_data = train_data.dropna(subset=["target"])
print(f"Train dataset: {train_data.shape}, {len(train_failed)} wells failed, "
      f"built in {time.time()-t0:.1f}s")

X_train = train_data[FEATURE_COLS]
y_train = train_data["target"].to_numpy()

stage4a_model = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
stage4a_model.fit(X_train, y_train)
print("Stage 4a component trained on", len(train_data), "rows.")

GLOBAL_FALLBACK = float(train_data["target"].mean())
print(f"GLOBAL_FALLBACK = {GLOBAL_FALLBACK:.2f}")

In [ ]:
test_wells = list_wells(TEST_DIR)
print(f"Building TEST features for {len(test_wells)} wells...")
test_data, test_failed = build_dataset(TEST_DIR, test_wells, has_target=False)
print(f"Test dataset: {test_data.shape}, {len(test_failed)} wells failed")

rows = []
if len(test_data) > 0:
    X_test = test_data[FEATURE_COLS]
    try:
        stage4a_pred = stage4a_model.predict(X_test)
        blend = (
            BLEND_WEIGHTS["linear_prior"] * test_data["linear_prior"].to_numpy()
            + BLEND_WEIGHTS["windowed_match"] * test_data["windowed_match"].to_numpy()
            + BLEND_WEIGHTS["stage4a_pred"] * stage4a_pred
        )
    except Exception as exc:  # noqa: BLE001
        print(f"blend predict failed ({exc}); falling back to GLOBAL_FALLBACK for all test rows")
        blend = np.full(len(test_data), GLOBAL_FALLBACK)

    for well, row_idx, pred in zip(test_data["well"], test_data["row_idx"], blend):
        rows.append({"id": f"{well}_{row_idx}", "tvt": pred})

submission = pd.DataFrame(rows, columns=["id", "tvt"])
print(f"built {len(submission)} predictions across "
      f"{len(test_wells) - len(test_failed)} wells ({len(test_failed)} wells failed)")
submission.head()

In [ ]:
sample = pd.read_csv(SAMPLE_SUBMISSION)
submission = submission.drop_duplicates(subset="id", keep="first")

merged = sample[["id"]].merge(submission, on="id", how="left")
n_missing = merged["tvt"].isna().sum()
if n_missing > 0:
    print(f"WARNING: {n_missing} required ids had no prediction - filling with "
          f"GLOBAL_FALLBACK ({GLOBAL_FALLBACK:.2f})")
    merged["tvt"] = merged["tvt"].fillna(GLOBAL_FALLBACK)

extra_ids = set(submission["id"]) - set(sample["id"])
if extra_ids:
    print(f"NOTE: {len(extra_ids)} predicted ids aren't in sample_submission - dropped")

assert len(merged) == len(sample), f"row count still off: {len(merged)} vs {len(sample)}"
assert merged["tvt"].notna().all(), "still have NaNs after fallback fill - should be impossible"

submission = merged
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv:", submission.shape)
print(f"Total runtime: {time.time()-t0:.1f}s")